Підготовка та завантаження даних

In [6]:
import pandas as pd
import numpy as np
import timeit
import os
from sklearn.preprocessing import MinMaxScaler, StandardScaler

def load_and_clean_data(relative_path):
    path = os.path.join(os.getcwd(), relative_path)
    
    if not os.path.exists(path):
        print(f"Помилка: Файл за шляхом {path} не знайдено!")
        return None

    df = pd.read_csv(path, sep=';', na_values='?', low_memory=False)

    df = df.dropna()

    numeric_columns = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                       'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col])
        
    return df

start_time = timeit.default_timer()
df = load_and_clean_data('data/household_power_consumption.txt')
end_time = timeit.default_timer()

if df is not None:
    print(f"Дані успішно завантажені за {end_time - start_time:.2f} секунд.")
    print(f"Розмір датасету: {df.shape}")
    display(df.head())

Дані успішно завантажені за 3.05 секунд.
Розмір датасету: (2049280, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


1. Записи, у яких загальна активна потужність перевищує 5 кВт

In [10]:
def task_1_high_power(data):
    return data[data['Global_active_power'] > 5.0]

t1 = timeit.timeit(lambda: task_1_high_power(df), number=1)
print(f"Завдання 1 виконано за: {t1:.5f} с. Знайдено рядків: {len(task_1_high_power(df))}")

Завдання 1 виконано за: 0.00464 с. Знайдено рядків: 17547


2. Струм 19-20 А, де пральна машина/холодильник (G2) > бойлер/кондиціонер (G3)

In [11]:
def task_2_current_filter(data):
    return data[(data['Global_intensity'] >= 19) & 
                (data['Global_intensity'] <= 20) & 
                (data['Sub_metering_2'] > data['Sub_metering_3'])]

t2 = timeit.timeit(lambda: task_2_current_filter(df), number=1)
print(f"Завдання 2 виконано за: {t2:.5f} с. Знайдено рядків: {len(task_2_current_filter(df))}")

Завдання 2 виконано за: 0.01650 с. Знайдено рядків: 2509


3. Випадкові 500,000 записів та середні величини груп споживання

In [12]:
def task_3_random_sample(data):
    sample = data.sample(n=500000, replace=False)
    means = sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return sample, means
    
t3 = timeit.timeit(lambda: task_3_random_sample(df), number=1)
_, res3 = task_3_random_sample(df)
print(f"Завдання 3 виконано за: {t3:.5f} с.\nСередні значення:\n{res3}")

Завдання 3 виконано за: 0.11413 с.
Середні значення:
Sub_metering_1    1.121244
Sub_metering_2    1.283208
Sub_metering_3    6.454854
dtype: float64


4. Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити...

In [14]:
def task_4_complex_slice(data):
    condition = (data['Time'] > '18:00:00') & \
                (data['Global_active_power'] > 6.0) & \
                (data['Sub_metering_2'] > data['Sub_metering_1']) & \
                (data['Sub_metering_2'] > data['Sub_metering_3'])
    
    filtered_df = data[condition]

    mid = len(filtered_df) // 2
    first_half = filtered_df.iloc[:mid:3]
    second_half = filtered_df.iloc[mid::4]
    
    return pd.concat([first_half, second_half])

t4 = timeit.timeit(lambda: task_4_complex_slice(df), number=1)
print(f"Завдання 4 виконано за: {t4:.5f} с. Результат: {len(task_4_complex_slice(df))} рядків.")

Завдання 4 виконано за: 0.11232 с. Результат: 310 рядків.


5. Пронормувати та стандартизувати вибраний датасет

In [24]:
def task_5_scaling(data):
    scaler_minmax = MinMaxScaler()
    scaler_std = StandardScaler()
    
    cols = ['Global_active_power', 'Global_intensity']
    
    normalized = scaler_minmax.fit_transform(data[cols])
    standardized = scaler_std.fit_transform(data[cols])
    
    return normalized, standardized

start_t5 = timeit.default_timer()
norm_data, std_data = task_5_scaling(df)
end_t5 = timeit.default_timer()

print(f"Завдання 5 виконано за: {end_t5 - start_t5:.5f} с.")
print(f"Нормовані дані (перші 2 рядки):\n{norm_data[:2]}")
print(f"Стандартизовані дані (перші 2 рядки):\n{std_data[:2]}")

Завдання 5 виконано за: 0.15216 с.
Нормовані дані (перші 2 рядки):
[[0.37479631 0.37759336]
 [0.47836321 0.47302905]]
Стандартизовані дані (перші 2 рядки):
[[2.95507706 3.09878851]
 [4.03708463 4.13379998]]


6. Коефіцієнти Пірсона та Спірмена

In [26]:
def task_6_correlation(data):
    p_val = data['Global_active_power'].corr(data['Voltage'], method='pearson')
    s_val = data['Global_active_power'].corr(data['Voltage'], method='spearman')
    return p_val, s_val

start_t6 = timeit.default_timer()
pearson_val, spearman_val = task_6_correlation(df)
end_t6 = timeit.default_timer()

print(f"Завдання 6 виконано за: {end_t6 - start_t6:.5f} с.")
print(f"Коефіцієнт Пірсона: {pearson_val:.4f}")
print(f"Коефіцієнт Спірмена: {spearman_val:.4f}")

Завдання 6 виконано за: 0.46047 с.
Коефіцієнт Пірсона: -0.3998
Коефіцієнт Спірмена: -0.3252


7. One Hot Encoding (OHE)

In [27]:
def task_7_ohe(data):
    data['Usage_Type'] = np.where(data['Global_active_power'] > 3.0, 'High', 'Normal')
    return pd.get_dummies(data['Usage_Type'], prefix='Usage')

start_t7 = timeit.default_timer()
ohe_results = task_7_ohe(df)
end_t7 = timeit.default_timer()

print(f"Завдання 7 виконано за: {end_t7 - start_t7:.5f} с.")
print("Результат One Hot Encoding (перші 5 рядків):")
display(ohe_results.head())

Завдання 7 виконано за: 0.35774 с.
Результат One Hot Encoding (перші 5 рядків):


,Usage_High,Usage_Normal
0,True,False
1,True,False
2,True,False
3,True,False
4,True,False
